In [ ]:
dem_fn = "labelC/N13E103_edem_egm_3C.tif"
idw_fn = "labelC/N13E103_edem_egm_3C_idw.tif"
aidw_fn = "labelC/N13E103_edem_egm_3C_aidw.tif"


In [ ]:
import os 
interp_dir = 'interp_dir'
os.makedirs(interp_dir, exist_ok=True)


In [5]:
import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import json
import os

def load_raster_data(raster_path):
    """Loads raster data and transforms pixel coordinates to geographic."""
    with rasterio.open(raster_path) as src:
        z_values = src.read(1)
        profile = src.profile
        metadata = src.tags()

        rows, cols = z_values.shape
        xs_pixel, ys_pixel = np.meshgrid(np.arange(cols), np.arange(rows))
        geo_xs, geo_ys = src.xy(ys_pixel.flatten(), xs_pixel.flatten())

        nodata_val = src.nodata
        
        return geo_xs, geo_ys, z_values.flatten(), profile, metadata, nodata_val

def filter_nodata(geo_xs, geo_ys, zs, nodata_val):
    """Filters out NoData values from XYZ arrays."""
    if nodata_val is not None:
        valid_mask = zs != nodata_val
        print(f"Filtered out {len(zs) - np.sum(valid_mask)} NoData pixels.")
        return geo_xs[valid_mask], geo_ys[valid_mask], zs[valid_mask]
    else:
        print("No NoData value defined. All pixels included.")
        return geo_xs, geo_ys, zs

def save_xyz_to_txt(file_path, x_coords, y_coords, z_values):
    """Saves X, Y, Z data to a tab-separated TXT file."""
    df = pd.DataFrame({'x': x_coords, 'y': y_coords, 'z': z_values})
    df.to_csv(file_path, sep='\t', index=False, header=False)
    print(f"XYZ data saved to: {file_path}")

def save_metadata_to_json(file_path, profile, metadata_tags):
    """Saves raster profile and metadata tags to a JSON file."""
    full_metadata = {'profile': profile, 'tags': metadata_tags}
    with open(file_path, 'w') as f:
        json.dump(full_metadata, f, indent=4)
    print(f"Profile and metadata saved to: {file_path}")

def create_geodataframe(x_coords, y_coords, z_values, crs):
    """Creates a GeoDataFrame from X, Y, Z coordinates and a CRS."""
    geometry = [Point(xy) for xy in zip(x_coords, y_coords)]
    gdf = gpd.GeoDataFrame(pd.DataFrame({'z': z_values}), geometry=geometry, crs=crs)
    print("GeoDataFrame created.")
    return gdf

def save_geodataframe(gdf, base_path, driver_name):
    """Saves a GeoDataFrame to specified file format (GPKG or SHP)."""
    file_path = f"{base_path}.{driver_name.lower()}"
    os.makedirs(os.path.dirname(file_path), exist_ok=True) # Ensure output directory exists
    gdf.to_file(file_path, driver=driver_name)
    print(f"GeoDataFrame saved to {driver_name}: {file_path}")

def process_raster(raster_path, out_dir=None, include_nodata_in_vector=False):
    """
    Main function to process a raster: extract XYZ, save as TXT,
    extract metadata, save as JSON, convert XYZ to GeoDataFrame,
    and save as GPKG and SHP.

    Args:
        raster_path (str): Path to the input GeoTIFF raster file.
        out_dir (str, optional): Directory where output files will be saved.
                                 If None, outputs are saved in the same directory as the raster.
        include_nodata_in_vector (bool, optional): If True, NoData pixels are
                                                  included in TXT and vector files.
                                                  Defaults to False (NoData filtered out).
    """
    base_name = os.path.splitext(os.path.basename(raster_path))[0]
    
    # Determine the output directory
    if out_dir is None:
        output_dir = os.path.dirname(raster_path) or '.' # Default to current directory if no path
    else:
        output_dir = out_dir
    
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    xyz_txt_path = os.path.join(output_dir, f"{base_name}.txt")
    metadata_json_path = os.path.join(output_dir, f"{base_name}.json")
    gpkg_path_base = os.path.join(output_dir, base_name)
    shp_path_base = os.path.join(output_dir, base_name)

    print(f"Processing raster: {raster_path}")
    print(f"Saving outputs to: {os.path.abspath(output_dir)}")

    try:
        geo_xs, geo_ys, zs, profile, metadata_tags, nodata_val = load_raster_data(raster_path)

        if not include_nodata_in_vector:
            geo_xs, geo_ys, zs = filter_nodata(geo_xs, geo_ys, zs, nodata_val)
        else:
            print("Including NoData pixels in XYZ and GeoDataFrame.")

        save_xyz_to_txt(xyz_txt_path, geo_xs, geo_ys, zs)
        save_metadata_to_json(metadata_json_path, profile, metadata_tags)

        gdf = create_geodataframe(geo_xs, geo_ys, zs, profile['crs'])

        save_geodataframe(gdf, gpkg_path_base, "GPKG")
        save_geodataframe(gdf, shp_path_base, "ESRI Shapefile")

        print("Processing complete.")

    except rasterio.errors.RasterioIOError as e:
        print(f"Error: Could not open or read raster file '{raster_path}'. {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [6]:
dem_fn = "/home/ljp238/Documents/uoe/TLS_PEX_LabCa/labelC/N13E103_edem_egm_3C.tif"

In [7]:
process_raster(raster_path=dem_fn, out_dir=interp_dir, include_nodata_in_vector=False)

Processing raster: /home/ljp238/Documents/uoe/TLS_PEX_LabCa/labelC/N13E103_edem_egm_3C.tif
Saving outputs to: /home/ljp238/Documents/uoe/TLS_PEX_LabCa/interp_dir
Filtered out 29233208 NoData pixels.
XYZ data saved to: interp_dir/N13E103_edem_egm_3C.txt
An unexpected error occurred: Object of type Profile is not JSON serializable
